# exp064_train_test_well_id_assert_probe train

Public-sample sanity notebook for the train/test `well_id` assert probe. This experiment does not train a model.

## Contents

1. Setup and configuration
2. Public sample overlap sanity
3. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from well_id_assert_probe import run_assert_probe, write_json

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
probe_params = config["model"]["params"]

print("Experiment:", EXPERIMENT_NAME)
print("Route:", config["experiment"]["route"])
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Expected public sample wells:", probe_params["expected_public_test_wells"])
print("Debug:", DEBUG)

## 2. Public sample overlap sanity


In [ ]:
probe_summary = run_assert_probe(
    train_dir=paths.train_data_dir,
    test_dir=paths.test_data_dir,
    expected_public_test_wells=probe_params["expected_public_test_wells"],
    suffix=probe_params["horizontal_suffix"],
)

print(json.dumps(probe_summary, indent=2, sort_keys=True))

## 3. Metrics and artifacts


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": "public_sample_sanity_completed",
    "created_at": datetime.now(UTC).isoformat(),
    "route": config["experiment"]["route"],
    "cv": None,
    "public_lb": None,
    "private_lb": None,
    "metric": None,
    "probe": probe_summary,
    "notes": "No model training. This train notebook only verifies the known public sample overlap handling.",
}

write_json(paths.metrics_path, metrics)
write_json(paths.artifacts_dir / "public_sample_overlap_sanity.json", probe_summary)
print("Metrics written:", paths.metrics_path)
print("Sanity artifact written:", paths.artifacts_dir / "public_sample_overlap_sanity.json")